# 02. 데이터 점검과 검증 분할

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jennk2025/HAIKUTHON/blob/main/notebooks/02_data_audit_split.ipynb)

**GPU가 필요 없다.** 런타임 유형을 **CPU**로 두고 실행한다(GPU 세션 한도를 아낀다).

| 단계 | 내용 |
|---|---|
| 점검 | ID 대응, 앞자리 0, 중복 ID, 빈 설명, 누락·손상 이미지, 형식·크기 분포, 완전 중복, 지각 해시 근접 중복 후보, 텍스트 유사도 |
| 분할 | 스타터 키트와 같은 seed 42 80/20 분할 + 확정 중복 그룹을 같은 쪽에 배치 |
| 비교 | 저장소에 커밋된 `splits/split_seed42.json`과 데이터 지문·ID 목록이 같은지 확인 |

원본 데이터는 읽기만 한다. 결과는 `RESULTS_DIR/audit/`에 저장한다(저장소 파일은 덮어쓰지 않음).

In [ ]:
# 저장소 받기 + 경로 설정 (Colab / Kaggle 공통). 노트북마다 맨 처음 한 번 실행한다.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/jennk2025/HAIKUTHON.git"
import importlib.util
IN_COLAB = importlib.util.find_spec("google.colab") is not None   # Colab에도 /kaggle 폴더가 있을 수 있어 Colab을 먼저 확인
IN_KAGGLE = (not IN_COLAB) and (bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or Path("/kaggle/input").exists())
if IN_COLAB or IN_KAGGLE:
    REPO_DIR = Path("/content/HAIKUTHON" if IN_COLAB else "/kaggle/working/HAIKUTHON")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
    else:                                    # 이미 받았으면 최신 코드로 갱신
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=False)
else:                                        # 로컬: notebooks/ 의 상위 폴더 = 저장소
    REPO_DIR = Path.cwd().resolve().parent
sys.path.insert(0, str(REPO_DIR))
print("REPO_DIR =", REPO_DIR)

In [ ]:
%pip install -q -r {REPO_DIR}/requirements-colab.txt

In [ ]:
# 데이터 + 스타터 키트 준비.
# Colab: Drive에 데이터 묶음이 있으면 그걸 풀고, 없으면 HF에서 받은 뒤 Drive에 저장한다(처음 한 번).
# Kaggle: 오른쪽 Settings에서 Internet을 켜야 HF에서 받을 수 있다.
from haiku.setup_data import prepare, env_info
P = prepare(use_drive=True, save_to_drive=True)
DATA_DIR, KIT_DIR, WORK_DIR, RESULTS_DIR = P["data_dir"], P["kit_dir"], P["work_dir"], P["results_dir"]

In [ ]:
import os, json
from haiku.audit import audit
AUDIT_DIR = RESULTS_DIR / "audit"
summary = audit(DATA_DIR, AUDIT_DIR, workers=os.cpu_count())
print(f"문제 {len(summary['issues'])}건, {summary['runtime']['seconds']}초")

In [ ]:
r = subprocess.run([sys.executable, "-m", "haiku.split", "--data-dir", str(DATA_DIR),
                    "--audit", str(AUDIT_DIR / "data_audit.json"),
                    "--out-dir", str(AUDIT_DIR / "splits"), "--report-dir", str(AUDIT_DIR)],
                   cwd=REPO_DIR, capture_output=True, text=True)
print(r.stdout, r.stderr)

# 저장소에 커밋된 분할과 비교
repo_split = json.loads((REPO_DIR / "splits" / "split_seed42.json").read_text(encoding="utf-8"))
new_split = json.loads((AUDIT_DIR / "splits" / "split_seed42.json").read_text(encoding="utf-8"))
print("데이터 지문 같음:", repo_split["data_fingerprint"] == new_split["data_fingerprint"])
print("검증 ID 같음:", repo_split["valid_ids"] == new_split["valid_ids"])

In [ ]:
from IPython.display import Markdown, Image as IPImage, display
display(Markdown((AUDIT_DIR / "data_audit.md").read_text(encoding="utf-8")))
display(Markdown((AUDIT_DIR / "split_report.md").read_text(encoding="utf-8")))
if (AUDIT_DIR / "near_dup_image_pairs.png").exists():
    display(IPImage(filename=str(AUDIT_DIR / "near_dup_image_pairs.png"), width=480))